## Load Libraries


In [1]:
# Data manipulation
import pandas as pd

# Scikit-learn for model selection and evaluation
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report

# Scikit-learn for various machine learning models
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

# Scikit-learn for preprocessing
from sklearn.preprocessing import LabelEncoder

## Load Dataset


In [4]:
data = pd.read_csv('/content/drive/MyDrive/FYP-Taskora/productivity_dataset_2.csv')
df = pd.DataFrame(data)

print(df.head())


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/FYP-Taskora/productivity_dataset_2.csv'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Data Cleaning and Preprocessing

**Reasoning**:

The part requires defining X and y, creating a text cleaning function, applying it, combining cleaned text with labels, and deduplicating the data. This code block will perform all these steps sequentially to ensure data integrity and prepare the data for further processing.



In [ ]:
import re
import nltk
from nltk.corpus import stopwords

# Ensure stopwords are downloaded
try:
    stopwords.words('english')
except LookupError:
    nltk.download('stopwords')

# 1. Define X and y
X = df['text']
y = df['label']

print(f"Original shape of X: {X.shape}")
print(f"Original shape of y: {y.shape}")

# 2. Create a function to clean text
def clean_text(text):
    # a. Lowercase
    text = text.lower()

    # b. Remove special characters but KEEP apostrophes (can't, I'm, I'll)
    text = re.sub(r"[^a-z0-9\s']", '', text)

    # c. Remove numbers
    text = re.sub(r'\d+', '', text)

    # d. Strip extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# 3. Apply the clean_text function to X to create X_cleaned
X_cleaned = X.apply(clean_text)
print(f"\nShape of X_cleaned after cleaning: {X_cleaned.shape}")

# 4. Create a new DataFrame df_cleaned by combining X_cleaned and y
df_cleaned = pd.DataFrame({'text_cleaned': X_cleaned, 'label': y})
print(f"Shape of df_cleaned: {df_cleaned.shape}")

# 5. Identify and remove duplicate rows
original_duplicates = df_cleaned.duplicated(subset=['text_cleaned']).sum()
df_cleaned_dedup = df_cleaned.drop_duplicates(subset=['text_cleaned']).reset_index(drop=True)
cleaned_duplicates = original_duplicates # This will be the number of duplicates removed

print(f"\nNumber of original duplicates based on cleaned text: {original_duplicates}")
print(f"Shape of df_cleaned_dedup after removing duplicates: {df_cleaned_dedup.shape}")

# 6. Separate df_cleaned_dedup back into X_deduplicated and y_deduplicated
X_deduplicated = df_cleaned_dedup['text_cleaned']
y_deduplicated = df_cleaned_dedup['label']

print(f"\nShape of X_deduplicated: {X_deduplicated.shape}")
print(f"Shape of y_deduplicated: {y_deduplicated.shape}")

## Real-World Dataset Integration

Integrates GoEmotions (Google) and DAIR-AI Emotion datasets to expose the model to genuine human language patterns. Mapped labels are merged with the synthetic dataset and capped per label to maintain class balance.

In [ ]:
# Install HuggingFace datasets library
!pip install datasets -q

In [ ]:
from datasets import load_dataset

print('Loading GoEmotions dataset...')
go_emotions = load_dataset('google-research-datasets/go_emotions', 'simplified')

print('Loading DAIR-AI Emotion dataset...')
dair_emotion = load_dataset('dair-ai/emotion')

print('\nBoth datasets loaded successfully.')

In [ ]:
# ── GoEmotions label mapping ────────────────────────────────────
# Maps GoEmotions label indices to 8 productivity labels of the my dataset.
# Only semantically aligned mappings are used.
# Label index reference: https://huggingface.co/datasets/go_emotions
go_emotions_map = {
    9:  'LOW_ENERGY',               # exhaustion
    26: 'LOW_ENERGY',               # tiredness
    17: 'WORK_OVERLOAD',            # overwhelmed
    15: 'HIGH_MOTIVATION',          # excitement
    19: 'HIGH_MOTIVATION',          # optimism
    20: 'CONSISTENT_PRODUCTIVITY',  # pride
    11: 'DISTRACTION',              # annoyance (external interruptions)
    6:  'POOR_PLANNING',            # confusion
    14: 'FORGETFULNESS',            # disappointment (missed something)
    18: 'PROCRASTINATION',          # nervousness (avoidance-related)
}

go_rows = []
for split in ['train', 'validation', 'test']:
    for row in go_emotions[split]:
        for label_id in row['labels']:
            if label_id in go_emotions_map:
                go_rows.append({
                    'text':  row['text'],
                    'label': go_emotions_map[label_id]
                })
                break  # take only first matching label per row

df_go = pd.DataFrame(go_rows)
print(f'GoEmotions mapped rows : {len(df_go)}')
print(df_go['label'].value_counts())

In [ ]:
# ── DAIR-AI Emotion label mapping ───────────────────────────────
# DAIR-AI labels: 0=sadness, 1=joy, 2=love, 3=anger, 4=fear, 5=surprise
# love(2) and surprise(5) are skipped — no clean mapping to your labels
dair_map = {
    0: 'LOW_ENERGY',       # sadness → drained / low energy
    1: 'HIGH_MOTIVATION',  # joy → excited / motivated
    3: 'WORK_OVERLOAD',    # anger → frustration from overload
    4: 'PROCRASTINATION',  # fear → avoidance / scared to start
}

dair_rows = []
for split in ['train', 'validation', 'test']:
    for row in dair_emotion[split]:
        if row['label'] in dair_map:
            dair_rows.append({
                'text':  row['text'],
                'label': dair_map[row['label']]
            })

df_dair = pd.DataFrame(dair_rows)
print(f'DAIR-AI mapped rows    : {len(df_dair)}')
print(df_dair['label'].value_counts())

In [ ]:
# ── Merge, clean and cap real-world data ────────────────────────

# Combine both real-world sources
df_real = pd.concat([df_go, df_dair], ignore_index=True)
df_real = df_real.drop_duplicates(subset=['text']).reset_index(drop=True)
print(f'Total real-world rows (before cleaning) : {len(df_real)}')

# Apply the same clean_text used on your synthetic data
df_real['text'] = df_real['text'].apply(clean_text)
df_real = df_real.drop_duplicates(subset=['text']).reset_index(drop=True)
print(f'Total real-world rows (after cleaning)  : {len(df_real)}')

# Cap at 300 per label — prevents real data from drowning synthetic data
REAL_CAP_PER_LABEL = 300
df_real_capped = (
    df_real
    .groupby('label', group_keys=False)
    .apply(lambda x: x.sample(min(len(x), REAL_CAP_PER_LABEL), random_state=42))
    .reset_index(drop=True)
)
print(f'\nReal-world samples after capping (max {REAL_CAP_PER_LABEL}/label):')
print(df_real_capped['label'].value_counts())

# Rename column to match synthetic dataset before merging
df_real_capped = df_real_capped.rename(columns={'text': 'text_cleaned'})

# Merge with your existing deduplicated synthetic dataset
df_combined = pd.concat([df_cleaned_dedup, df_real_capped], ignore_index=True)
df_combined = df_combined.drop_duplicates(subset=['text_cleaned']).reset_index(drop=True)

print(f'\nSynthetic samples       : {len(df_cleaned_dedup)}')
print(f'Real-world samples added: {len(df_real_capped)}')
print(f'Final combined dataset  : {len(df_combined)} samples')
print(f'\nLabel distribution in combined dataset:')
print(df_combined['label'].value_counts())

**Reasoning**:
The subtask explicitly mentions 'encoding labels' which was not fully addressed in the previous step (only the column was named 'label' instead of 'label_encoded', and the values themselves are still strings). To complete the preprocessing, I will now encode the categorical labels in `y_deduplicated` into numerical representations using `LabelEncoder` and update the dataframe `df_cleaned_dedup` with the encoded labels.



In [ ]:
from sklearn.model_selection import train_test_split

# Use combined dataset (synthetic + real-world)
X_combined = df_combined['text_cleaned']
y_combined = df_combined['label']

X_train, X_test, y_train, y_test = train_test_split(
    X_combined, y_combined,
    test_size=0.2,
    random_state=42,
    stratify=y_combined   # keeps label distribution balanced in both splits
)

print(f'Training samples : {X_train.shape[0]}')
print(f'Testing samples  : {X_test.shape[0]}')

# TF-IDF vectorization — refit on combined training data
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(ngram_range=(1, 2), max_features=5000)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)  # transform only, never fit on test

print(f'\nTF-IDF matrix shape (train): {X_train_tfidf.shape}')
print(f'TF-IDF matrix shape (test) : {X_test_tfidf.shape}')

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
import pandas as pd

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "SVC":                 SVC(kernel="linear", probability=True),
    "Random Forest":       RandomForestClassifier(n_estimators=200, random_state=42)
}

results = {}

for name, model in models.items():
    print(f"\n{'═'*52}")
    print(f"  Training: {name}")
    print(f"{'═'*52}")

    # Train
    model.fit(X_train_tfidf, y_train)

    # Predict
    y_pred = model.predict(X_test_tfidf)

    # Score
    acc = accuracy_score(y_test, y_pred)
    results[name] = acc

    print(f"  Accuracy : {acc:.4f}")
    print(f"\n{classification_report(y_test, y_pred)}")

# Summary
print(f"\n{'═'*52}")
print("  SUMMARY")
print(f"{'═'*52}")
for name, acc in sorted(results.items(), key=lambda x: x[1], reverse=True):
    bar = "█" * int(acc * 40)
    print(f"  {name:<25} {acc:.4f}  {bar}")
print(f"{'═'*52}")

In [ ]:
# data taskora api might say
real_world_tests = [
    ("I couldn't get id one", "PROCRASTINATION"),
    ("my head is pounding and i have no juice left", "LOW_ENERGY"),
    ("everyone keeps asking me stuff and i cant get anything done", "DISTRACTION"),
    ("i had no idea this would take so long", "POOR_PLANNING"),
    ("i completely blanked on that task sorry", "FORGETFULNESS"),
    ("i am absolutely smashing my goals rn", "HIGH_MOTIVATION"),
    ("there is way too much coming at me from all directions", "WORK_OVERLOAD"),
    ("knocked everything out ahead of time feeling good", "CONSISTENT_PRODUCTIVITY"),
]

print("Real-world generalization test")
print(f"{'═'*60}")

# Use Logistic Regression (fastest)
model = models["Logistic Regression"]
correct = 0

for text, true_label in real_world_tests:
    cleaned   = clean_text(text)
    vec       = tfidf.transform([cleaned])
    predicted = model.predict(vec)[0]
    proba     = model.predict_proba(vec).max()
    status    = "✔" if predicted == true_label else "✘"
    if predicted == true_label:
        correct += 1
    print(f"  {status} [{proba:.2f}] {predicted:<28} ← {text[:45]}")

print(f"{'═'*60}")
print(f"  Score: {correct}/{len(real_world_tests)} real-world sentences correct")

In [ ]:
import joblib

# Save the best model (swap to SVC/RF if it scored higher)
joblib.dump(models["Logistic Regression"], "taskora_classifier.pkl")
joblib.dump(tfidf, "taskora_tfidf.pkl")

print("Model and vectorizer saved!")